# RVC Colab 训练入口

这个 notebook 会 clone 你的 RVC 仓库，并使用仓库内的 `rvc_training_data/colab/train_rvc_colab.py` 执行完整训练流程：预处理、F0、HuBERT 特征、训练和 index 构建。

请先把本地生成的 30-60 分钟、单条 5-15 秒数据集 zip 上传到 Google Drive。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 参数

`DATASET_ZIP` 和 `DATASET_DIR` 二选一：如果上传的是 zip，填写 `DATASET_ZIP`；如果已经是解压后的目录，填写 `DATASET_DIR`。

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/rikaaa0928/Retrieval-based-Voice-Conversion-WebUI.git'
REPO_DIR = Path('/content/RVC')
EXPERIMENT = 'leijun_zh_v2_48k'

# zip 示例: /content/drive/MyDrive/rvc_datasets/zh_leijun_30m.zip
DATASET_ZIP = Path('/content/drive/MyDrive/rvc_datasets/zh_leijun_30m.zip')
DATASET_DIR = Path('/content/drive/MyDrive/rvc_datasets/zh_leijun_30m')

SAMPLE_RATE = '48k'
VERSION = 'v2'
IF_F0 = 1
F0_METHOD = 'rmvpe'
GPUS = '0'
BATCH_SIZE = 8
TOTAL_EPOCH = 200
SAVE_EVERY_EPOCH = 20

EXPORT_DIR = Path('/content/drive/MyDrive/rvc_models') / EXPERIMENT

## 安装 RVC

Notebook 会直接 clone `REPO_URL` 指向的你的仓库；其中已经包含训练脚本，不需要单独上传 `train_rvc_colab.py`。Colab Python 3.11+ 会自动使用 `requirements-py311.txt`。

In [ ]:
%cd /content
if REPO_DIR.exists():
    print(f'Reusing existing repo: {REPO_DIR}')
else:
    !git clone "$REPO_URL" "$REPO_DIR"
%cd /content/RVC
import sys
requirements_file = 'requirements-py311.txt' if sys.version_info >= (3, 11) else 'requirements.txt'
print(f'Python: {sys.version.split()[0]}, installing {requirements_file}')
!pip install -r "$requirements_file"
!python tools/download_models.py

## 准备训练脚本

训练脚本已经随 `REPO_URL` 仓库 clone 到 Colab，本单元只做存在性检查。

In [ ]:
SCRIPT_TARGET = REPO_DIR / 'rvc_training_data' / 'colab' / 'train_rvc_colab.py'
if not SCRIPT_TARGET.exists():
    raise FileNotFoundError(f'仓库中缺少训练脚本: {SCRIPT_TARGET}')
print(f'Training script: {SCRIPT_TARGET}')

## 准备数据集

In [ ]:
import zipfile

if DATASET_ZIP.exists():
    DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATASET_ZIP) as archive:
        archive.extractall(DATASET_DIR.parent)

if not DATASET_DIR.exists():
    raise FileNotFoundError(f'找不到数据集目录: {DATASET_DIR}')

audio_dir = DATASET_DIR / 'audio'
if not audio_dir.exists():
    audio_dir = DATASET_DIR

audio_files = [p for p in audio_dir.iterdir() if p.suffix.lower() in {'.mp3', '.wav', '.flac', '.m4a', '.ogg', '.aac', '.opus'}]
print(f'Dataset: {DATASET_DIR}')
print(f'Audio files: {len(audio_files)}')
if not audio_files:
    raise RuntimeError('数据集目录里没有音频文件。')

## 开始训练

In [ ]:
!python "$SCRIPT_TARGET" \
  --repo-dir "$REPO_DIR" \
  --dataset-dir "$DATASET_DIR" \
  --experiment "$EXPERIMENT" \
  --sample-rate "$SAMPLE_RATE" \
  --version "$VERSION" \
  --if-f0 "$IF_F0" \
  --f0-method "$F0_METHOD" \
  --gpus "$GPUS" \
  --batch-size "$BATCH_SIZE" \
  --total-epoch "$TOTAL_EPOCH" \
  --save-every-epoch "$SAVE_EVERY_EPOCH" \
  --save-latest 1 \
  --cache-gpu 0 \
  --save-every-weights 1

## 导出到 Drive

In [ ]:
import glob
import shutil

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
for pattern in [
    str(REPO_DIR / 'assets' / 'weights' / f'{EXPERIMENT}.pth'),
    str(REPO_DIR / 'logs' / EXPERIMENT / '*.index'),
    str(REPO_DIR / 'logs' / EXPERIMENT / 'train.log'),
    str(REPO_DIR / 'logs' / EXPERIMENT / 'colab_train_summary.json'),
]:
    for src in glob.glob(pattern):
        shutil.copy2(src, EXPORT_DIR / Path(src).name)

print(f'Exported files to: {EXPORT_DIR}')
print('\n'.join(str(p) for p in sorted(EXPORT_DIR.iterdir())))